# PitchLogic — Colab GPU 临时后端

**运行顺序：** Cell1 检查GPU → Cell2 装依赖 → Cell3 建目录 → Cell4 上传权重 → Cell5 克隆代码 → Cell6 写后端 → Cell7 启动 → Cell8 保活

In [8]:
# Cell 1: 检查 GPU
import subprocess
r = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(r.stdout if r.returncode == 0 else '⚠️  No GPU — switch Runtime to T4/A100')


Fri Mar 27 07:41:53 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   41C    P8             13W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [9]:
# Cell 2: 安装依赖（首次约3-5分钟）
%pip install -q flask flask-cors ultralytics supervision \
    opencv-python-headless numpy pandas scikit-learn scipy \
    matplotlib decord seaborn Pillow loguru iopath tqdm lmdb \
    hydra-core jpeg4py \
    'git+https://github.com/roboflow/sports.git'

import subprocess, os
if not os.path.exists('/usr/local/bin/cloudflared'):
    subprocess.run([
        'wget', '-q',
        'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64',
        '-O', '/usr/local/bin/cloudflared'
    ], check=True)
    subprocess.run(['chmod', '+x', '/usr/local/bin/cloudflared'], check=True)
print('✅ 依赖安装完成')


  Preparing metadata (setup.py) ... done
✅ 依赖安装完成


In [10]:
# Cell 3: 创建目录结构
import os
for d in [
    '/content/pitchlogic/backend/app/pipeline',
    '/content/pitchlogic/backend/app/routes',
    '/content/pitchlogic/backend/weights/football',
    '/content/pitchlogic/backend/weights/keypoints',
    '/content/pitchlogic/outputs',
    '/content/pitchlogic/uploads',
]:
    os.makedirs(d, exist_ok=True)
print('✅ 目录结构创建完成')


✅ 目录结构创建完成


In [11]:
# Cell 4: 挂载 Drive 并复制模型权重
from google.colab import drive
drive.mount('/content/drive')

import shutil, os
shutil.copy(
    '/content/drive/MyDrive/samurai_env/samurai/weight_football/best.pt',
    '/content/pitchlogic/backend/weights/football/best.pt'
)
shutil.copy(
    '/content/drive/MyDrive/samurai_env/samurai/weights_keypoints/best.pt',
    '/content/pitchlogic/backend/weights/keypoints/best.pt'
)

fb = os.path.exists('/content/pitchlogic/backend/weights/football/best.pt')
kb = os.path.exists('/content/pitchlogic/backend/weights/keypoints/best.pt')
print('Football model:', '✅' if fb else '❌ 缺失')
print('Keypoint model:', '✅' if kb else '❌ 缺失')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Football model: ✅
Keypoint model: ✅


In [12]:
# Cell 5: 从 GitHub 克隆后端代码
import subprocess, shutil, os

# 克隆仓库
if not os.path.exists('/content/AI_football_analizer'):
    subprocess.run(
        ['git', 'clone', 'https://github.com/Muriki1234/AI_football_analizer.git',
         '/content/AI_football_analizer'],
        check=True
    )
else:
    print('仓库已存在，跳过克隆')

# 复制 backend/app 和 config.py
shutil.copytree('/content/AI_football_analizer/backend/app',
                '/content/pitchlogic/backend/app',
                dirs_exist_ok=True)
shutil.copy('/content/AI_football_analizer/backend/config.py',
            '/content/pitchlogic/backend/config.py')

# 验证关键文件
for f in [
    'app/pipeline/session_manager.py',
    'app/pipeline/tasks.py',
    'app/pipeline/analysis_core.py',
    'config.py',
]:
    ok = os.path.exists(f'/content/pitchlogic/backend/{f}')
    print(f'{"✅" if ok else "❌"} {f}')


仓库已存在，跳过克隆
✅ app/pipeline/session_manager.py
✅ app/pipeline/tasks.py
✅ app/pipeline/analysis_core.py
✅ config.py


In [13]:
%%writefile /content/colab_backend.py
import os, sys, threading
from pathlib import Path
from flask import Flask, request, jsonify, send_file, make_response
import torch

sys.path.insert(0, '/content/pitchlogic/backend')
sys.path.insert(0, '/content/drive/MyDrive/samurai_env/samurai/sam2')
os.environ.setdefault('YOLO_MODEL_PATH',     '/content/pitchlogic/backend/weights/football/best.pt')
os.environ.setdefault('KEYPOINT_MODEL_PATH', '/content/pitchlogic/backend/weights/keypoints/best.pt')
os.environ.setdefault('SAMURAI_SCRIPT',      '/content/drive/MyDrive/samurai_env/samurai/scripts/demo.py')

from app.pipeline.session_manager import SessionManager
from app.pipeline import tasks

app = Flask(__name__)
OUTPUT_ROOT = Path('/content/pitchlogic/outputs')
UPLOAD_ROOT = Path('/content/pitchlogic/uploads')
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
UPLOAD_ROOT.mkdir(parents=True, exist_ok=True)
sm = SessionManager(output_root=OUTPUT_ROOT)

def _fire(fn, *args):
    threading.Thread(target=fn, args=args, daemon=True).start()

def _s404(sid):
    s = sm.get_session(sid)
    return (s, None) if s else (None, (jsonify({"error": "Session not found"}), 404))

@app.before_request
def handle_options():
    if request.method == "OPTIONS":
        r = make_response()
        r.headers["Access-Control-Allow-Origin"] = "*"
        r.headers["Access-Control-Allow-Headers"] = "Content-Type, Authorization, X-Requested-With"
        r.headers["Access-Control-Allow-Methods"] = "GET, POST, PUT, DELETE, OPTIONS"
        return r

@app.after_request
def add_cors(response):
    response.headers["Access-Control-Allow-Origin"] = "*"
    response.headers["Access-Control-Allow-Headers"] = "Content-Type, Authorization, X-Requested-With"
    response.headers["Access-Control-Allow-Methods"] = "GET, POST, PUT, DELETE, OPTIONS"
    return response

@app.route("/health")
def health():
    gpu = {"available": False, "name": "CPU only"}
    if torch.cuda.is_available():
        p = torch.cuda.get_device_properties(0)
        gpu = {"available": True, "name": torch.cuda.get_device_name(0),
               "memory_gb": round(p.total_memory/1e9,1),
               "free_gb": round((p.total_memory-torch.cuda.memory_allocated(0))/1e9,1)}
    return jsonify({"status": "ok", "gpu": gpu})

@app.route("/api/<sid>/receive_video", methods=["POST"])
def receive_video(sid):
    if "file" in request.files:
        f = request.files["file"]
        dest = UPLOAD_ROOT / f"{sid}_{f.filename}"
        f.save(str(dest))
        video_path = str(dest)
    else:
        data = request.get_json(silent=True) or {}
        video_path = data.get("video_path", "")
        if not os.path.exists(video_path):
            return jsonify({"error": f"video_path not found: {video_path}"}), 400
    if not sm.get_session(sid):
        sm.create_session(sid, video_path)
        sm.session_output_dir(sid).mkdir(parents=True, exist_ok=True)
    else:
        sm.update_status(sid, "uploaded", stage="video_received")
    print(f"✅ [receive_video] sid={sid} path={video_path}")
    return jsonify({"status": "received", "session_id": sid})

@app.route("/api/<sid>/analyze_frame", methods=["POST"])
def analyze_frame(sid):
    session, err = _s404(sid)
    if err: return err
    import cv2
    from ultralytics import YOLO
    data = request.get_json(silent=True) or {}
    time_s = float(data.get("time_in_seconds", 0))
    video_path = session["video_path"]
    if not os.path.exists(video_path):
        return jsonify({"error": f"Video not found: {video_path}"}), 404
    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS) or 24
    cap.set(cv2.CAP_PROP_POS_FRAMES, int(time_s * fps))
    ret, frame = cap.read()
    cap.release()
    if not ret:
        return jsonify({"error": "Extract frame failed"}), 500
    model = YOLO(os.environ.get("YOLO_MODEL_PATH", "weights/football/best.pt"))
    results = model.predict(frame, conf=0.25, verbose=False)[0]
    players = []
    for box in results.boxes:
        cls = int(box.cls[0])
        if results.names[cls].lower() in ("player", "person", "referee", "goalkeeper"):
            x1, y1, x2, y2 = box.xyxy[0].tolist()
            players.append({"id": len(players)+1, "name": results.names[cls].capitalize(),
                            "bbox": [x1,y1,x2,y2], "avatar": "👤"})
    print(f"✅ [analyze_frame] sid={sid} 检测到 {len(players)} 人")
    return jsonify({"players_data": players,
                    "image_dimensions": {"width": frame.shape[1], "height": frame.shape[0]}})

@app.route("/api/<sid>/trim", methods=["POST"])
def trim_video(sid):
    session, err = _s404(sid)
    if err: return err
    import subprocess as sp
    data = request.get_json(silent=True) or {}
    start_t = float(data.get("start", 0))
    end_t   = float(data.get("end", 0))
    original_path = session["video_path"]
    out_path = os.path.join(os.path.dirname(original_path), f"trimmed_{sid}.mp4")
    print(f"✂️ [trim] sid={sid} {start_t}s -> {end_t}s")
    sp.run(["ffmpeg", "-y", "-i", original_path,
            "-ss", str(start_t), "-to", str(end_t), "-c", "copy", out_path],
           stdout=sp.DEVNULL, stderr=sp.DEVNULL)
    session["video_path"] = out_path
    return jsonify({"status": "success", "trimmed_path": out_path})

@app.route("/api/<sid>/track", methods=["POST"])
def start_tracking(sid):
    session, err = _s404(sid)
    if err: return err
    if session["status"] not in ("uploaded", "tracking_failed"):
        return jsonify({"error": f"Cannot track from: {session['status']}"}), 400
    body = request.get_json(silent=True) or {}
    if "x1" in body:
        x1,y1,x2,y2 = int(body["x1"]),int(body["y1"]),int(body["x2"]),int(body["y2"])
        bbox = {"x":x1,"y":y1,"w":x2-x1,"h":y2-y1}
    elif all(k in body for k in ("x","y","w","h")):
        bbox = {k:int(body[k]) for k in ("x","y","w","h")}
    else:
        return jsonify({"error": "Missing bbox"}), 400
    bbox["frame"] = int(body.get("frame", 0))
    sm.update_status(sid, "tracking", progress=0, stage="samurai_init")
    _fire(tasks.run_samurai_tracking, sid, session, bbox, sm)
    return jsonify({"status": "tracking_started"})

@app.route("/api/<sid>/analyze", methods=["POST"])
def start_analysis(sid):
    session, err = _s404(sid)
    if err: return err
    if session["status"] != "tracking_done":
        return jsonify({"error": f"Tracking must complete first. Current: {session['status']}"}), 400
    sm.update_status(sid, "analyzing", progress=0, stage="yolo_global")
    _fire(tasks.run_global_analysis, sid, session, sm)
    return jsonify({"status": "analysis_started"})

GENERATE_MAP = {
    "heatmap":        tasks.run_heatmap,
    "speed_chart":    tasks.run_speed_chart,
    "possession":     tasks.run_possession_stats,
    "minimap_replay": tasks.run_minimap_replay,
    "full_replay":    tasks.run_full_replay,
}

@app.route("/api/<sid>/generate/<gen_type>", methods=["POST"])
def generate(sid, gen_type):
    session, err = _s404(sid)
    if err: return err
    if session["status"] != "analysis_done":
        return jsonify({"error": f"Analysis not done. Status: {session['status']}"}), 400
    fn = GENERATE_MAP.get(gen_type)
    if not fn:
        return jsonify({"error": f"Unknown type: {gen_type}"}), 400
    task_id = sm.create_task(sid, gen_type)
    _fire(fn, sid, session, task_id, sm)
    return jsonify({"task_id": task_id, "status": "queued"})

STAGE_LABELS = {
    "samurai_running":    "SAMURAI tracking...",
    "yolo_detection":     "YOLO detection...",
    "camera_motion":      "Camera motion compensation...",
    "keypoint_detection": "Keypoint detection...",
    "ball_interpolation": "Ball trajectory...",
    "speed_calculation":  "Speed & distance...",
    "team_assignment":    "Team colors...",
    "computing_summary":  "Generating summary...",
    "done":               "Done",
}

def _available(s):
    st = s["status"]
    if st == "analysis_done": return ["heatmap","speed_chart","possession","minimap_replay","full_replay","summary"]
    if st == "tracking_done": return ["analyze"]
    if st == "uploaded":      return ["track"]
    return []

@app.route("/api/<sid>/status")
def get_status(sid):
    session, err = _s404(sid)
    if err: return err
    return jsonify({
        "session_id":         sid,
        "status":             session["status"],
        "progress":           session.get("progress", 0),
        "stage":              session.get("stage", ""),
        "stage_label":        STAGE_LABELS.get(session.get("stage",""), session.get("stage","")),
        "error":              session.get("error"),
        "available_features": _available(session),
    })

@app.route("/api/<sid>/task/<task_id>")
def task_status(sid, task_id):
    t = sm.get_task(sid, task_id)
    return (jsonify(t), 200) if t else (jsonify({"error": "Task not found"}), 404)

@app.route("/api/<sid>/summary")
def summary(sid):
    session, err = _s404(sid)
    if err: return err
    s = session.get("player_summary")
    return (jsonify(s), 200) if s else (jsonify({"error": "Not available"}), 404)

@app.route("/api/<sid>/file/<task_id>")
def serve_file(sid, task_id):
    t = sm.get_task(sid, task_id)
    if not t or t["status"] != "done":
        return jsonify({"error": "File not ready"}), 404
    fp = t.get("file_path")
    if not fp or not os.path.exists(fp):
        return jsonify({"error": "File missing"}), 404
    return send_file(fp, conditional=True)

if __name__ == "__main__":
    app.run(host="0.0.0.0", port=7860, debug=False, use_reloader=False)

Writing /content/colab_backend.py


In [14]:
# Cell 7: 启动 Flask + cloudflared 隧道
import threading, time, os, requests, subprocess, re

def run_flask():
    os.system('python /content/colab_backend.py')

threading.Thread(target=run_flask, daemon=True).start()

# 等待 Flask 启动
public_url = None
for i in range(10):
    time.sleep(2)
    try:
        r = requests.get('http://localhost:7860/health', timeout=3)
        print(f'✅ Flask OK: {r.json()["gpu"]["name"]}')
        break
    except:
        print(f'⏳ 等待 Flask... ({(i+1)*2}s)')

# 启动隧道
tunnel = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', 'http://localhost:7860'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
)
print('⏳ 等待隧道...')
for _ in range(60):
    line = tunnel.stdout.readline()
    m = re.search(r'https://[\w-]+\.trycloudflare\.com', line)
    if m:
        public_url = m.group(0)
        break

print()
print('='*60)
print('🎉 COLAB 后端已就绪！')
print(f'URL: {public_url}')
print('='*60)
print(f'健康检查: {public_url}/health')




⏳ 等待 Flask... (2s)
⏳ 等待 Flask... (4s)
⏳ 等待 Flask... (6s)
⏳ 等待 Flask... (8s)
⏳ 等待 Flask... (10s)
⏳ 等待 Flask... (12s)
⏳ 等待 Flask... (14s)
⏳ 等待 Flask... (16s)
⏳ 等待 Flask... (18s)
✅ Flask OK: Tesla T4
⏳ 等待隧道...

🎉 COLAB 后端已就绪！
URL: https://district-aspect-jones-man.trycloudflare.com
健康检查: https://district-aspect-jones-man.trycloudflare.com/health


In [ ]:
# Cell 8: 保活（防止 Colab 自动断开）
import requests, threading, time

def heartbeat():
    while True:
        try:
            r = requests.get('http://localhost:7860/health', timeout=5)
            name = r.json().get('gpu', {}).get('name', '?')
            print(f'💓 heartbeat OK — {name}', flush=True)
        except Exception as e:
            print(f'⚠️ heartbeat failed: {e}', flush=True)
        time.sleep(300)

threading.Thread(target=heartbeat, daemon=True).start()
print('✅ 保活已启动（5分钟心跳）')
print()
print('浏览器端保活，复制到 DevTools Console：')
print('setInterval(()=>document.dispatchEvent(new MouseEvent("mousemove",{bubbles:true})),60000)')


In [ ]:
 !ls -d /content/drive/MyDrive/samurai_env/samurai/sam2

/content/drive/MyDrive/samurai_env/samurai/sam2
